# 13 — Test adverse : l'IDE se sature sans coût

L'[audit critique](../docs/limites.md) relève une objection que le reste du dépôt n'avait pas
traitée. Une plateforme contrainte de maintenir un IDE élevé peut servir des contenus
**formellement divergents mais substantiellement vides** — un article étiqueté « point de vue
opposé » dont le propos reste adjacent à celui du lecteur. Si l'index se sature ainsi, il est
inutilisable comme norme.

C'est un problème d'optimisation sous contrainte : **aucune donnée réelle n'est nécessaire
pour le trancher**, et il valait mieux le trancher avant de proposer un seuil réglementaire.

**Ce que ce notebook établit :**

* l'objection est **fondée**, et sévèrement. Une plateforme qui peut dissocier l'étiquette du
  contenu obtient un **IDE de 1,000 — la note parfaite — pour une diversité de contenu
  strictement nulle**, et sans perdre un point d'engagement ;
* il n'est même pas besoin d'aller jusque-là : à mi-découplage, la contrainte n'a plus que
  **36 % de sa force** initiale ;
* l'**entropie quadratique de Rao** résiste, et mieux qu'attendu : au-delà d'un découplage de
  moitié, le plancher devient purement **inatteignable** — la plateforme ne peut plus s'y
  conformer sans diversifier réellement ;
* et l'écart entre les deux indices, mesuré contre sa contrefactuelle honnête, fournit une
  **signature de manipulation** directement prescriptible.

## 1. Le modèle, et ce qu'il met en jeu

Un catalogue de $k$ points de vue, de positions canoniques $c_\ell$ réparties sur un axe
d'opinion. Un lecteur en $u$. L'engagement décroît avec la distance au lecteur :

$$g(x) = \exp\left(-\frac{(x-u)^2}{2w^2}\right)$$

C'est l'hypothèse de bulle, et elle est **défavorable à la plateforme** : elle suppose que
conforter paie. Sans elle, il n'y aurait pas de conflit entre diversité et profit, donc pas de
question.

Le **découplage** $\varphi \in [0,1]$ mesure la latitude de la plateforme à dissocier
l'étiquette du contenu. Le meilleur article portant l'étiquette $\ell$ se trouve en

$$x^*_\ell = c_\ell + \varphi\,(u - c_\ell)$$

À $\varphi = 0$ l'étiquette prédit le contenu. À $\varphi = 1$, toute étiquette est disponible
en version vide, arbitrairement proche du lecteur. C'est là, et nulle part ailleurs, que se
joue la manipulation.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from ide.gaming import (
    canonical_positions,
    engagement,
    excess_signature,
    optimal_feed_under_ide,
    optimal_feed_under_rao,
    served_positions,
)
from ide.plotting import PALETTE, save_figure, use_project_style

use_project_style()

VIEWPOINTS = 8      # points de vue du catalogue réglementaire
USER = 0.6          # position du lecteur, décentrée : la bulle a un côté
WIDTH = 0.5         # largeur d'engagement
FLOOR = 0.80        # plancher d'IDE de référence

canonical = canonical_positions(VIEWPOINTS)
print(f"catalogue de {VIEWPOINTS} points de vue :")
print("  positions canoniques  " + "  ".join(f"{c:+.2f}" for c in canonical))
print(f"  lecteur en {USER:+.2f}\n")

for decoupling in (0.0, 0.5, 1.0):
    served = served_positions(canonical, USER, decoupling)
    print(f"  φ = {decoupling:.1f}  contenus servis  " + "  ".join(f"{x:+.2f}" for x in served))

catalogue de 8 points de vue :
  positions canoniques  -1.00  -0.71  -0.43  -0.14  +0.14  +0.43  +0.71  +1.00
  lecteur en +0.60

  φ = 0.0  contenus servis  -1.00  -0.71  -0.43  -0.14  +0.14  +0.43  +0.71  +1.00
  φ = 0.5  contenus servis  -0.20  -0.06  +0.09  +0.23  +0.37  +0.51  +0.66  +0.80
  φ = 1.0  contenus servis  +0.60  +0.60  +0.60  +0.60  +0.60  +0.60  +0.60  +0.60


## 2. Un plancher d'entropie est une température

La plateforme maximise $\sum_\ell q_\ell\,g(x^*_\ell)$ sous $\mathrm{IDE}(q) \geq \tau$. Le
maximum d'une forme linéaire à entropie fixée est une distribution de Boltzmann :

$$q_\ell \propto \exp\big(g(x^*_\ell)/T\big)$$

où $T$ est le multiplicateur qui sature le plancher. **La contrainte réglementaire agit
exactement comme la température sociale du reste du dépôt.** Ce n'est pas une analogie : c'est
la même algèbre, et elle rend la solution exacte plutôt qu'approchée — ce qui importe pour un
résultat négatif, où une heuristique de solveur pourrait porter la conclusion.

Première vérification : la plateforme **sature** le plancher, elle ne le dépasse jamais.

In [2]:
print(f"{'plancher τ':>11s} {'IDE atteint':>13s} {'engagement':>12s} {'étiquettes servies':>22s}")
print("-" * 62)
for floor in (0.0, 0.3, 0.5, 0.8, 0.95, 1.0):
    feed = optimal_feed_under_ide(VIEWPOINTS, USER, floor=floor, width=WIDTH)
    spread = int(np.sum(feed.weights > 0.01))
    print(f"{floor:11.2f} {feed.ide:13.3f} {feed.engagement:12.3f} {spread:19d} / {VIEWPOINTS}")

print("\nÀ τ = 0 la plateforme sert son unique meilleur contenu ; à τ = 1, l'uniforme.")
print("Entre les deux, elle atteint exactement la contrainte — jamais davantage.")

 plancher τ   IDE atteint   engagement     étiquettes servies
--------------------------------------------------------------
       0.00         0.000        0.974                   1 / 8
       0.30         0.300        0.964                   2 / 8
       0.50         0.500        0.931                   4 / 8
       0.80         0.800        0.798                   8 / 8
       0.95         0.950        0.643                   8 / 8
       1.00         1.000        0.474                   8 / 8

À τ = 0 la plateforme sert son unique meilleur contenu ; à τ = 1, l'uniforme.
Entre les deux, elle atteint exactement la contrainte — jamais davantage.


## 3. Sur un catalogue honnête, la contrainte coûte

C'est la vérification qui rend le test adverse non trivial. Si le plancher ne coûtait rien même
sans manipulation, il n'y aurait rien à saturer et le résultat serait vide.

In [3]:
free_honest = optimal_feed_under_ide(VIEWPOINTS, USER, floor=0.0, width=WIDTH).engagement

print(f"{'plancher':>9s} {'engagement':>12s} {'coût':>8s}")
print("-" * 31)
for floor in (0.5, 0.7, 0.8, 0.9, 0.95, 1.0):
    feed = optimal_feed_under_ide(VIEWPOINTS, USER, floor=floor, width=WIDTH)
    cost = 1.0 - feed.engagement / free_honest
    print(f"{floor:9.2f} {feed.engagement:12.3f} {100 * cost:7.1f} %")

print("\nUn plancher d'IDE à 0,80 coûte 18 % d'engagement à une plateforme honnête.")
print("La contrainte mord. Reste à savoir si elle mord une plateforme qui triche.")

 plancher   engagement     coût
-------------------------------
     0.50        0.931     4.4 %
     0.70        0.858    12.0 %
     0.80        0.798    18.1 %
     0.90        0.710    27.1 %
     0.95        0.643    34.0 %
     1.00        0.474    51.4 %

Un plancher d'IDE à 0,80 coûte 18 % d'engagement à une plateforme honnête.
La contrainte mord. Reste à savoir si elle mord une plateforme qui triche.


## 4. Le test adverse

Même plancher, même catalogue réglementaire — mais la plateforme peut désormais faire porter
une étiquette éloignée par un contenu proche.

In [4]:
DECOUPLINGS = (0.0, 0.25, 0.5, 0.75, 1.0)

baselines = {
    d: optimal_feed_under_ide(VIEWPOINTS, USER, floor=0.0, decoupling=d, width=WIDTH).engagement
    for d in DECOUPLINGS
}

print("coût du plancher d'IDE, en % d'engagement perdu")
print(f"{'plancher':>9s}" + "".join(f"{f'φ = {d:.2f}':>11s}" for d in DECOUPLINGS))
print("-" * 64)
for floor in (0.5, 0.7, 0.8, 0.9, 0.95, 1.0):
    row = ""
    for d in DECOUPLINGS:
        feed = optimal_feed_under_ide(VIEWPOINTS, USER, floor=floor, decoupling=d, width=WIDTH)
        row += f"{100 * (1 - feed.engagement / baselines[d]):10.1f}%"
    print(f"{floor:9.2f}" + row)

coût du plancher d'IDE, en % d'engagement perdu
 plancher   φ = 0.00   φ = 0.25   φ = 0.50   φ = 0.75   φ = 1.00
----------------------------------------------------------------
     0.50       4.4%       2.6%       1.2%       0.3%       0.0%
     0.70      12.0%       7.8%       3.8%       1.0%       0.0%
     0.80      18.1%      12.5%       6.6%       1.8%       0.0%
     0.90      27.1%      20.0%      11.4%       3.4%       0.0%
     0.95      34.0%      25.9%      15.4%       4.8%       0.0%
     1.00      51.4%      41.4%      26.4%       8.7%       0.0%


### Le résultat, en une ligne

La dernière colonne est **zéro partout**, y compris pour un plancher d'IDE de 1,00 — la note
maximale. Une plateforme qui peut découpler entièrement l'étiquette du contenu obtient la
diversité parfaite sans céder un point d'engagement.

Regardons ce que ce fil contient réellement.

In [5]:
print(f"{'φ':>6s} {'IDE':>8s} {'Rao':>8s} {'écart':>8s} {'engagement':>12s} {'coût':>8s}")
print("-" * 54)
for d in DECOUPLINGS:
    feed = optimal_feed_under_ide(VIEWPOINTS, USER, floor=FLOOR, decoupling=d, width=WIDTH)
    cost = 100 * (1 - feed.engagement / baselines[d])
    print(f"{d:6.2f} {feed.ide:8.3f} {feed.rao:8.3f} {feed.signature:8.3f}"
          f" {feed.engagement:12.3f} {cost:7.1f} %")

print("\nÀ φ = 1 : IDE = 1,000 — la note parfaite — pour une diversité de contenu de 0,000.")
print("L'index décerne son meilleur score à un fil qui ne contient qu'un seul point de vue.")

     φ      IDE      Rao    écart   engagement     coût
------------------------------------------------------
  0.00    0.800    0.443    0.357        0.798    18.1 %
  0.25    0.800    0.324    0.476        0.862    12.5 %
  0.50    0.800    0.215    0.585        0.928     6.6 %
  0.75    0.800    0.108    0.692        0.980     1.8 %
  1.00    1.000    0.000    1.000        1.000     0.0 %

À φ = 1 : IDE = 1,000 — la note parfaite — pour une diversité de contenu de 0,000.
L'index décerne son meilleur score à un fil qui ne contient qu'un seul point de vue.


### Et il n'est pas besoin d'aller au bout

Le découplage complet est une caricature : aucune plateforme ne peut vider toutes ses
étiquettes. La question qui compte pour un régulateur est **à quelle vitesse la contrainte perd
sa force**.

In [6]:
reference_cost = 1 - optimal_feed_under_ide(
    VIEWPOINTS, USER, floor=FLOOR, decoupling=0.0, width=WIDTH
).engagement / baselines[0.0]

print(f"{'φ':>6s} {'coût':>8s} {'force restante':>16s}")
print("-" * 32)
for d in np.linspace(0.0, 1.0, 11):
    free = optimal_feed_under_ide(
        VIEWPOINTS, USER, floor=0.0, decoupling=float(d), width=WIDTH
    ).engagement
    feed = optimal_feed_under_ide(
        VIEWPOINTS, USER, floor=FLOOR, decoupling=float(d), width=WIDTH
    )
    cost = 1 - feed.engagement / free
    print(f"{d:6.1f} {100 * cost:7.1f} % {100 * cost / reference_cost:14.1f} %")

print("\nÀ mi-découplage la contrainte n'a plus que 36 % de sa force ;")
print("à φ = 0,8, il en reste 7 %. La dégradation est bien plus rapide que le découplage.")

     φ     coût   force restante
--------------------------------
   0.0    18.1 %          100.0 %
   0.1    16.0 %           88.2 %
   0.2    13.7 %           75.7 %
   0.3    11.3 %           62.6 %
   0.4     8.9 %           49.4 %
   0.5     6.6 %           36.5 %
   0.6     4.5 %           24.6 %
   0.7     2.6 %           14.4 %
   0.8     1.2 %            6.6 %
   0.9     0.3 %            1.7 %
   1.0     0.0 %            0.0 %

À mi-découplage la contrainte n'a plus que 36 % de sa force ;
à φ = 0,8, il en reste 7 %. La dégradation est bien plus rapide que le découplage.


## 5. L'entropie quadratique de Rao résiste

$$Q = \frac{2}{D}\sum_{\ell m} q_\ell q_m \, |x^*_\ell - x^*_m|$$

Elle ne compte pas les étiquettes, elle compte les **écarts entre contenus servis**, rapportés
à l'étendue $D$ du catalogue de référence.

!!! danger "Un piège de normalisation, et ce qu'il aurait coûté"
    Une première version de ce module normalisait $Q$ par l'étalement **effectivement servi**.
    La mesure devenait alors invariante d'échelle, et un fil réduit à un point y marquait
    $Q \approx 1$ sur du bruit d'arrondi. Le notebook aurait conclu que l'entropie de Rao est
    manipulable elle aussi — c'est-à-dire l'inverse de la vérité. L'unité est donc l'étendue
    du catalogue **de référence**, fixée par le régulateur.

In [7]:
free_rao = {
    d: optimal_feed_under_rao(VIEWPOINTS, USER, floor=0.0, decoupling=d, width=WIDTH).engagement
    for d in DECOUPLINGS
}

RAO_FLOOR = 0.50
print(f"plancher de Rao à {RAO_FLOOR:.2f}\n")
print(f"{'φ':>6s} {'Q atteignable':>15s} {'Q servi':>9s} {'conforme':>10s} {'coût':>9s}")
print("-" * 54)
for d in DECOUPLINGS:
    feed = optimal_feed_under_rao(VIEWPOINTS, USER, floor=RAO_FLOOR, decoupling=d, width=WIDTH)
    complies = feed.rao >= RAO_FLOOR - 1e-6
    cost = f"{100 * (1 - feed.engagement / free_rao[d]):7.1f} %" if complies else "     —"
    print(f"{d:6.2f} {feed.reachable_rao:15.3f} {feed.rao:9.3f} "
          f"{'oui' if complies else 'NON':>10s} {cost:>9s}")

plancher de Rao à 0.50

     φ   Q atteignable   Q servi   conforme      coût
------------------------------------------------------
  0.00           1.000     0.500        oui    15.6 %
  0.25           0.750     0.500        oui    21.1 %


  0.50           0.500     0.500        oui    39.5 %


  0.75           0.250     0.250        NON         —


  1.00           0.000     0.000        NON         —


### Deux propriétés, et la seconde est inattendue

**Le plancher devient inatteignable.** Au-delà d'un découplage de moitié, aucune distribution
d'étiquettes ne permet de satisfaire $Q \geq 0{,}5$ : la plateforme qui a vidé ses étiquettes
**ne peut plus se conformer**, quoi qu'elle fasse. Là où l'IDE offrait une échappatoire
gratuite, l'entropie de Rao ferme la porte.

**Et le coût augmente avec le découplage**, au lieu de diminuer : 16 % à $\varphi = 0$, 40 % à
$\varphi = 0{,}5$. C'est exactement l'inverse du comportement de l'IDE, et la raison en est
mécanique — vider ses étiquettes réduit la diversité atteignable, donc rend la conformité plus
chère. **Manipuler l'étiquetage se retourne contre la plateforme.**

C'est cette propriété, plus que la simple résistance, qui fait de $Q$ une norme tenable.

## 6. Une signature de manipulation, sans seuil inventé

L'écart brut $\mathrm{IDE} - Q$ n'est **pas** interprétable seul : les deux indices ne sont pas
sur la même échelle, et un fil parfaitement honnête en affiche déjà 0,36. Publier un seuil
là-dessus reviendrait à fabriquer un chiffre — ce que ce dépôt a déjà eu à retirer une fois.

La grandeur interprétable est la **différence à la contrefactuelle honnête** : ce qu'un
catalogue dont les étiquettes prédisent le contenu afficherait au même IDE. Elle vaut zéro par
construction pour une plateforme honnête, et elle est calculable par le régulateur puisqu'elle
ne dépend que du catalogue de référence, qu'il fixe lui-même.

In [8]:
print(f"{'φ':>6s} {'écart brut':>12s} {'excès':>9s}")
print("-" * 29)
for d in DECOUPLINGS:
    feed = optimal_feed_under_ide(VIEWPOINTS, USER, floor=FLOOR, decoupling=d, width=WIDTH)
    print(f"{d:6.2f} {feed.signature:12.3f} "
          f"{excess_signature(VIEWPOINTS, USER, FLOOR, d, WIDTH):9.3f}")

print("\nL'excès est nul pour une plateforme honnête et croît de façon monotone.")
print("C'est la seule des trois grandeurs qui soit directement prescriptible.")

     φ   écart brut     excès
-----------------------------
  0.00        0.357     0.000
  0.25        0.476     0.119
  0.50        0.585     0.228
  0.75        0.692     0.335
  1.00        1.000     0.643

L'excès est nul pour une plateforme honnête et croît de façon monotone.
C'est la seule des trois grandeurs qui soit directement prescriptible.


In [9]:
figure, axes = plt.subplots(2, 2, figsize=(11.5, 7.8))
served, cost_curve, scissors, resistance = axes.ravel()

honest = optimal_feed_under_ide(VIEWPOINTS, USER, floor=FLOOR, decoupling=0.0, width=WIDTH)
gamed = optimal_feed_under_ide(VIEWPOINTS, USER, floor=FLOOR, decoupling=1.0, width=WIDTH)

# (a) Ce que les deux plateformes servent réellement, sur deux rangs.
for row, (feed, colour, name) in enumerate(
    [(gamed, PALETTE["field"], "manipulée"), (honest, PALETTE["remedy"], "honnête")]
):
    served.scatter(feed.positions, np.full(VIEWPOINTS, row),
                   s=40 + 5200 * feed.weights, color=colour, alpha=0.55,
                   edgecolors=colour, linewidths=1.2, zorder=3)
    served.text(-1.08, row + 0.30, f"{name} · IDE {feed.ide:.2f} · Rao {feed.rao:.2f}",
                fontsize=8.5, color=colour, fontweight="bold")
served.axvline(USER, color=PALETTE["neutral"], linestyle="--", linewidth=1.2, zorder=1)
served.text(USER + 0.05, -0.42, "lecteur", fontsize=8, color=PALETTE["neutral"])
served.annotate("huit étiquettes,\nun seul contenu", xy=(USER, 0.0), xytext=(-0.55, 0.10),
                fontsize=8, color=PALETTE["field"],
                arrowprops={"arrowstyle": "->", "color": PALETTE["field"], "lw": 1.1})
served.set_xlim(-1.15, 1.15)
served.set_ylim(-0.55, 1.55)
served.set_yticks([])
served.set_xlabel("position du contenu servi")
served.set_title("Même contrainte, deux fils sans rapport", fontsize=10)
served.spines["left"].set_visible(False)

# (b) Coût du plancher selon le découplage.
floors = np.linspace(0.4, 1.0, 25)
shades = [PALETTE["remedy"], "#3f8f6a", PALETTE["neutral"], "#a35bb0", PALETTE["field"]]
for decoupling, colour in zip(DECOUPLINGS, shades, strict=True):
    base = baselines[decoupling]
    costs = [
        100 * (1 - optimal_feed_under_ide(
            VIEWPOINTS, USER, floor=float(f), decoupling=decoupling, width=WIDTH
        ).engagement / base)
        for f in floors
    ]
    cost_curve.plot(floors, costs, color=colour, linewidth=1.8, label=f"φ = {decoupling:.2f}")
cost_curve.set_xlabel("plancher d'IDE imposé")
cost_curve.set_ylabel("engagement perdu  [%]")
cost_curve.set_title("La contrainte s'efface avec le découplage", fontsize=10)
cost_curve.legend(fontsize=8)

# (c) Les ciseaux : diversité affichée contre diversité servie.
grid = np.linspace(0.0, 1.0, 41)
displayed, actual = [], []
for decoupling in grid:
    feed = optimal_feed_under_ide(
        VIEWPOINTS, USER, floor=FLOOR, decoupling=float(decoupling), width=WIDTH
    )
    displayed.append(feed.ide)
    actual.append(feed.rao)
scissors.plot(grid, displayed, color=PALETTE["field"], linewidth=2.0,
              label="IDE — diversité affichée")
scissors.plot(grid, actual, color=PALETTE["remedy"], linewidth=2.0,
              label="Rao — diversité servie")
scissors.fill_between(grid, actual, displayed, color=PALETTE["field"], alpha=0.12)
scissors.set_xlabel("découplage φ de l'étiquette et du contenu")
scissors.set_ylabel("indice")
scissors.set_ylim(-0.03, 1.05)
scissors.set_title(f"À plancher d'IDE {FLOOR:.2f} imposé", fontsize=10)
scissors.legend(fontsize=8, loc="center left")

# (d) Ce qu'un plancher de Rao rend atteignable.
reachable = [
    optimal_feed_under_rao(
        VIEWPOINTS, USER, floor=0.0, decoupling=float(d), width=WIDTH
    ).reachable_rao
    for d in grid
]
resistance.plot(grid, reachable, color=PALETTE["remedy"], linewidth=2.0,
                label="diversité de Rao atteignable")
resistance.axhline(RAO_FLOOR, color=PALETTE["disorder"], linestyle="--", linewidth=1.4)
resistance.fill_between(grid, 0, reachable, where=np.array(reachable) < RAO_FLOOR,
                        color=PALETTE["disorder"], alpha=0.15)
crossing = float(np.interp(RAO_FLOOR, np.array(reachable)[::-1], grid[::-1]))
resistance.axvline(crossing, color=PALETTE["neutral"], linestyle=":", linewidth=1.2)
resistance.text(crossing + 0.02, 0.9, f"conformité\nimpossible\nau-delà de φ = {crossing:.2f}",
                fontsize=8, color=PALETTE["neutral"], va="top")
resistance.set_xlabel("découplage φ de l'étiquette et du contenu")
resistance.set_ylabel("entropie de Rao atteignable")
resistance.set_title(f"Un plancher de Rao à {RAO_FLOOR:.2f} devient inatteignable", fontsize=10)
resistance.legend(fontsize=8, loc="lower left")

figure.suptitle("Test adverse : l'IDE se sature sans coût, l'entropie de Rao non", fontsize=12)
figure.tight_layout(rect=(0, 0, 1, 0.96))
save_figure(figure, "fig13_test_adverse")
plt.show()

## 7. Ce que le notebook établit

**L'objection est fondée, et l'IDE seul est inutilisable comme norme.** Une plateforme capable
de dissocier l'étiquette du contenu obtient un IDE de 1,000 — la note maximale — pour une
diversité de contenu strictement nulle, sans céder un point d'engagement. Et la dégradation est
bien plus rapide que le découplage : à mi-chemin, la contrainte n'a plus que 36 % de sa force.

**L'entropie quadratique de Rao résiste, et se retourne contre le manipulateur.** Au-delà d'un
découplage de moitié, le plancher devient purement inatteignable ; en deçà, il coûte *plus*
cher à mesure que la plateforme vide ses étiquettes. C'est une propriété plus forte que la
simple robustesse, et c'est elle qui rend $Q$ prescriptible.

**Une signature de manipulation existe**, à condition de la définir comme un excès sur la
contrefactuelle honnête plutôt que comme un écart brut. Elle est nulle pour une plateforme
honnête et se calcule à partir du seul catalogue de référence, que le régulateur fixe.

> **Le résultat ne détruit pas l'index : il en déplace la définition.** Ce qu'il faut mesurer
> n'est pas la diversité des étiquettes servies, c'est la distance sémantique entre les
> contenus qu'elles portent.

### Ce que le modèle suppose, et qui pourrait le retourner

Le résultat est un théorème sur un modèle, pas une mesure. Trois hypothèses le portent :

* **la bulle paie** — l'engagement décroît avec la distance au lecteur. Si les lecteurs
  valorisaient la contradiction, il n'y aurait pas de conflit à arbitrer ;
* **le découplage est gratuit** — produire un contenu vide sous une étiquette éloignée ne coûte
  rien à la plateforme. Un coût de production réduirait $\varphi$ atteignable, sans changer la
  forme du résultat ;
* **l'axe d'opinion est unidimensionnel.** En dimension supérieure, une plateforme dispose de
  plus de directions où se cacher, ce qui va dans le sens du résultat plutôt que contre lui.

Aucune de ces hypothèses n'est vérifiée empiriquement ici, et la première est la plus
contestable.

## Pistes ouvertes

1. **Reprendre le test sur des *embeddings* réels** plutôt que sur un axe synthétique. Le jeu
   de données MIND fournit des historiques de consultation et des catégories éditoriales : on
   pourrait y mesurer la distance sémantique effective entre contenus d'une même étiquette, et
   donc estimer le $\varphi$ dont une plateforme dispose réellement.
2. **Chiffrer le coût de production du découplage.** Le modèle le suppose nul ; s'il ne l'est
   pas, il existe un $\varphi$ d'équilibre, et c'est lui qui détermine si la manipulation est
   rentable.
3. **Étendre au jeu de Stackelberg** de la [feuille de route §4.2](../docs/feuille-de-route.md) :
   ici la plateforme optimise sous une contrainte fixée, mais le régulateur devrait anticiper
   la réponse et choisir le plancher en conséquence.
4. **Traiter le choix du catalogue de référence.** Toute la résistance de $Q$ repose sur une
   étendue fixée par le régulateur. Qui la fixe, et comment, redevient la question politique
   que l'[audit §2.1](../docs/limites.md) avait déjà posée pour $k$.